# PAC-Mech-Interp: Interactive Demo

**Mechanistic Evidence of Layer-Dependent Positional Bias in In-Context Learning**

This notebook provides an interactive walkthrough of the two main contributions of the paper:

1. **Mechanistic Analysis**: Extracting and visualizing layer-wise attention weights from GPT-2 during ICL.
2. **PAC Intervention**: Running Position-Aware Calibration with different λ values.

---

In [ ]:
# Install dependencies if needed
# !pip install torch transformers matplotlib seaborn scipy tqdm

In [ ]:
import sys
sys.path.insert(0, '..')  # Add repo root to path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import SentimentICLDataset, TEST_QUERIES, LABEL_SPACE
from src.attention import AttentionExtractor
from src.pac import PACInference
from src.utils import compute_metrics, run_statistical_tests

print('Imports successful!')

## Part 1: Mechanistic Analysis

We load GPT-2 with `eager` attention to access raw attention weight tensors, then analyze how attention is distributed across demonstration positions in each layer.

In [ ]:
# Load the model
extractor = AttentionExtractor(model_name='gpt2', device='cpu')
dataset = SentimentICLDataset(k=4, n_pos_demo=2, n_neg_demo=2)

In [ ]:
# Build a representative 4-shot ICL prompt
demonstrations = dataset.demonstrations
query_text, true_label = TEST_QUERIES[0]
prompt = dataset.build_prompt(demonstrations, query_text)

print('=== ICL Prompt ===')
print(prompt)
print(f'\nTrue label: {true_label}')

In [ ]:
# Extract attention weights
layer_attention_mass, aggregated_mass = extractor.extract_attention_for_prompt(
    prompt, demonstrations
)

print('Layer-wise attention mass (rows=layers L0-L11, cols=positions Pos1-Pos4):')
print(np.round(layer_attention_mass, 3))
print(f'\nAggregated (mean across layers):')
for i, mass in enumerate(aggregated_mass):
    label = '(Farthest)' if i == 0 else ('(Closest)' if i == 3 else '')
    print(f'  Pos {i+1} {label}: {mass:.3f}')

In [ ]:
# Plot Figure 1(a): Attention heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Heatmap
pos_labels = ['Pos 1\n(Farthest)', 'Pos 2', 'Pos 3', 'Pos 4\n(Closest)']
layer_labels = [f'L{i}' for i in range(12)]

sns.heatmap(
    layer_attention_mass, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=pos_labels, yticklabels=layer_labels,
    ax=axes[0], vmin=0, vmax=0.8,
    cbar_kws={'label': 'Normalized Attention Mass'}
)
axes[0].set_title('(a) Attention Distribution by Layer', fontweight='bold')
axes[0].set_xlabel('Demonstration Position')
axes[0].set_ylabel('Transformer Layer')

# Bar chart
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
axes[1].bar(pos_labels, aggregated_mass, color=colors, width=0.5)
axes[1].axhline(y=0.25, color='gray', linestyle='--', label='Uniform (0.25)')
for i, v in enumerate(aggregated_mass):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 0.75)
axes[1].set_title('(b) Overall Attention by Position', fontweight='bold')
axes[1].set_ylabel('Normalized Attention Mass')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/demo_fig1_attention.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to figures/demo_fig1_attention.png')

## Part 2: PAC Inference

We now demonstrate PAC with different λ values on a single test query.

In [ ]:
# Load PAC inference engine
pac = PACInference(model_name='gpt2', device='cpu')

In [ ]:
# Test PAC on a single query with different lambda values
query_text, true_label = TEST_QUERIES[0]
print(f'Query: "{query_text}"')
print(f'True label: {true_label}\n')

for lam in [0.0, 0.3, 0.5, 0.7, 1.0]:
    pred = pac.predict_pac(demonstrations, query_text, lam)
    correct = '✓' if pred == true_label else '✗'
    print(f'  λ={lam:.1f}: Predicted={pred} {correct}')

In [ ]:
# Evaluate PAC across all 24 permutations for a single lambda
# Note: This runs 24 * 10 = 240 inferences — may take a few minutes on CPU
print('Evaluating λ=0.7 across all 24 permutations and 10 test queries...')
result_07 = pac.evaluate_all_permutations(TEST_QUERIES, lam=0.7)
print(f"  Mean accuracy:       {result_07['mean_accuracy']:.3f}  (paper: 0.833)")
print(f"  Worst-case accuracy: {result_07['worst_case_accuracy']:.3f}  (paper: 0.583)")
print(f"  Std deviation:       {result_07['std_dev']:.3f}  (paper: 0.124)")

In [ ]:
# Full lambda sweep (this will take ~10-20 minutes on CPU)
# Uncomment to run:
# sweep_results = pac.sweep_lambda(TEST_QUERIES)
# from src.utils import print_results_table
# print_results_table(sweep_results)

## Summary

This demo has shown:

1. **Layer-dependent positional bias**: Early layers (L0) show recency bias, while middle/late layers (L2-L11) show strong primacy bias. The net effect is that Position 1 (farthest) receives ~55% of total attention mass.

2. **PAC effectiveness**: By subtracting the content-free logits scaled by λ, PAC significantly improves mean accuracy from ~58% to ~83%.

3. **λ trade-off**: λ=0.7 maximizes mean accuracy; λ=1.0 (equivalent to CC) maximizes worst-case robustness.

For full reproduction of all paper figures, run:
```bash
bash experiments/run_all.sh
```